# SAGE Control UI via ngrok

This notebook clones the repo if needed, starts the real FastAPI server, protects the control UI with `SAGE_WEB_PASSWORD`, and exposes it through ngrok.

The public URL serves:

- `GET /` for the browser control panel
- `GET /health`
- `POST /generate` on the GPU server


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("SAGE_REPO_URL", "https://huggingface.co/sage002/sage")
REPO_ROOT = Path(os.environ.get("SAGE_REPO_ROOT", "/content/sage"))

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)


In [ ]:
%pip install -q -r requirements.txt pyngrok requests

In [ ]:
import os
import secrets

try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN")

if not NGROK_AUTHTOKEN:
    raise RuntimeError("Set NGROK_AUTHTOKEN in Colab Secrets or the environment before continuing.")

SAGE_WEB_PASSWORD = os.environ.get("SAGE_WEB_PASSWORD") or secrets.token_urlsafe(12)
os.environ["SAGE_WEB_PASSWORD"] = SAGE_WEB_PASSWORD

print("SAGE_WEB_PASSWORD:", SAGE_WEB_PASSWORD)


In [ ]:
import atexit
import subprocess
import sys
import time

import requests
import torch

HOST = "127.0.0.1"
PORT = 8000
APP_TARGET = "serve.server:app" if torch.cuda.is_available() else "serve.server_cpu:app"
SERVER_LOG = REPO_ROOT / "uvicorn.log"

SERVER_PROCESS = globals().get("SERVER_PROCESS")

def stop_server():
    global SERVER_PROCESS
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
    SERVER_PROCESS = None

stop_server()
server_log_handle = open(SERVER_LOG, "w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", APP_TARGET, "--host", HOST, "--port", str(PORT)],
    cwd=str(REPO_ROOT),
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)
atexit.register(stop_server)

health_url = f"http://{HOST}:{PORT}/health"
for _ in range(60):
    if SERVER_PROCESS.poll() is not None:
        server_log_handle.flush()
        raise RuntimeError(SERVER_LOG.read_text(encoding="utf-8", errors="ignore"))
    try:
        response = requests.get(health_url, timeout=2)
        if response.ok:
            print("Local server ready:", response.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    server_log_handle.flush()
    raise TimeoutError(SERVER_LOG.read_text(encoding="utf-8", errors="ignore"))

print("App target:", APP_TARGET)


In [ ]:
import atexit

from pyngrok import ngrok

NGROK_TUNNEL = globals().get("NGROK_TUNNEL")

def stop_tunnel():
    global NGROK_TUNNEL
    if NGROK_TUNNEL is not None:
        ngrok.disconnect(NGROK_TUNNEL.public_url)
        NGROK_TUNNEL = None

stop_tunnel()
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTHTOKEN)
NGROK_TUNNEL = ngrok.connect(addr=PORT, proto="http", bind_tls=True)
atexit.register(stop_tunnel)

print("Control UI:", NGROK_TUNNEL.public_url)
print("Health URL:", f"{NGROK_TUNNEL.public_url}/health")
if APP_TARGET == "serve.server:app":
    print("Generate URL:", f"{NGROK_TUNNEL.public_url}/generate")
print("Login password:", SAGE_WEB_PASSWORD)


In [ ]:
public_health = requests.get(f"{NGROK_TUNNEL.public_url}/health", timeout=20)
print(public_health.status_code)
print(public_health.json())


In [ ]:
# Optional cleanup
stop_tunnel()
stop_server()
server_log_handle.close()
